# Ellipse Array Generator (Nazca)

**Author:** Jason P. Beech  
**Date:** 2026-03  
**Affiliation:** Tegenfeldt Lab / Lund University  

---

This notebook generates an array of vertically stacked ellipses using **Nazca**.

## What the code does
- Each ellipse has a fixed width.
- Heights are determined from a list of aspect ratios.
- A vertical stack of ellipses forms one unit cell.
- That unit cell is repeated in an array.

## Main settings
At the top of the code cell, you can change:

- `EXPORT_MODE`
  - `"plot"` → preview only
  - `"gds"` → export GDS only
- `w_um` → ellipse width
- `max_ar` → maximum aspect ratio
- `N_ar` → number of ellipses per cell
- `W`, `H` → number of unit cells in x and y
- `space_um` → spacing between ellipses and cells

## Geometry notes
- Aspect ratio is defined as:

  \[
  \text{aspect ratio} = \frac{\text{height}}{\text{width}}
  \]

- Ellipse heights are computed as:

```python
height = aspect_ratio * w_um

In [2]:
# =============================================================================
# Ellipse Array Generator (Nazca)
#
# Author: Jason P. Beech
# Date: 2026-03
# Affiliation: Tegenfeldt Lab / Lund University
#
# Description:
# Generates an array of vertically stacked ellipses with varying aspect ratios.
# =============================================================================

import nazca as nd
import math
import numpy as np

# =============================================================================
# Output control
# =============================================================================
# Choose one of:
#   "plot" -> preview only
#   "gds"  -> export GDS only
EXPORT_MODE = "gds"


# =============================================================================
# Design parameters (units: µm)
# =============================================================================

w_um = 50                            # fixed ellipse width for all ellipses
space_um = round(0.5 * w_um)         # gap between ellipses and between cells

max_ar = 2.0                         # maximum aspect ratio
N_ar = 5                             # number of ellipses per cell

# Linearly spaced aspect ratios from 1.0 to max_ar
aspect_ratios = np.linspace(1.0, max_ar, N_ar).tolist()

# Polygon resolution used to approximate each ellipse
ellipse_points = 128

# Array size: W cells in x, H cells in y
W = 20
H = 4

# Optional outline around the full array
add_outline = False


# =============================================================================
# Export filename
# =============================================================================

from datetime import datetime

date_tag = datetime.now().strftime("%Y%m%d")

GDS_FILENAME = (
    f"ellipse_array_"
    f"{W}x{H}_"
    f"w{w_um:g}um_"
    f"AR{aspect_ratios[0]:g}-{aspect_ratios[-1]:g}_"
    f"N{len(aspect_ratios)}_"
    f"{date_tag}.gds"
)

# =============================================================================
# Helper function
# =============================================================================

def ellipse_bl_um(x0, y0, w, h, layer=1, n=ellipse_points):
    """
    Draw an ellipse as a closed polygon inside a bounding box.

    Parameters
    ----------
    x0, y0 : float
        Bottom-left corner of the ellipse bounding box.
    w, h : float
        Width and height of the ellipse bounding box in µm.
    layer : int
        GDS layer number.
    n : int
        Number of polygon points used to approximate the ellipse.
    """
    cx, cy = x0 + w / 2.0, y0 + h / 2.0
    rx, ry = w / 2.0, h / 2.0

    pts = [
        (cx + rx * math.cos(t), cy + ry * math.sin(t))
        for t in (2 * math.pi * i / n for i in range(n))
    ]
    pts.append(pts[0])  # close the polygon

    nd.Polygon(pts, layer=layer).put(0, 0)


# =============================================================================
# Derive ellipse heights from aspect ratios
# =============================================================================

if len(aspect_ratios) < 1:
    raise ValueError("aspect_ratios must have at least one value.")

# Aspect ratio is defined here as: height / width
# Nazca accepts floating-point coordinates, so heights can remain floats.
heights_um = [max(1.0, ar * w_um) for ar in aspect_ratios]
N = len(heights_um)

# Unit cell height = sum of ellipse heights + gaps between ellipses
unit_h_um = sum(heights_um) + space_um * (N - 1)

# Cell-to-cell pitch in x and y
pitchX_um = w_um + space_um
pitchY_um = unit_h_um + space_um


# =============================================================================
# Print summary
# =============================================================================

print(f"W x H = {W} x {H}")
print(f"N ellipses per cell = {N}")
print(f"Aspect ratios = {aspect_ratios}")
print(f"Ellipse heights (µm) = {heights_um}")
print(f"Unit cell height = {unit_h_um} µm")
print(f"pitchX = {pitchX_um} µm, pitchY = {pitchY_um} µm")


# =============================================================================
# Place array
# =============================================================================

for j in range(H):
    for i in range(W):
        x = i * pitchX_um
        y = j * pitchY_um

        y_cursor = y
        for h in heights_um:
            ellipse_bl_um(x, y_cursor, w_um, h, layer=1)
            y_cursor += h + space_um


# =============================================================================
# Optional outline
# =============================================================================

if add_outline:
    array_w_um = (W - 1) * pitchX_um + w_um
    array_h_um = (H - 1) * pitchY_um + unit_h_um
    nd.Polygon(
        [(0, 0), (array_w_um, 0), (array_w_um, array_h_um), (0, array_h_um), (0, 0)],
        layer=1
    ).put(0, 0)


# =============================================================================
# Export / preview
# =============================================================================

mode = EXPORT_MODE.lower()

if mode == "gds":
    nd.export_gds(filename=GDS_FILENAME)
    print(f"GDS exported: {GDS_FILENAME}")

elif mode == "plot":
    nd.export_plt()
    print("Plot exported.")

elif mode == "both":
    # Plot first, then export GDS.
    # In some Nazca setups, export_gds() can leave nothing active for export_plt().
    nd.export_plt()
    nd.export_gds(filename=GDS_FILENAME)
    print("Plot exported.")
    print(f"GDS exported: {GDS_FILENAME}")

else:
    raise ValueError("EXPORT_MODE must be one of: 'plot', 'gds', 'both'")

W x H = 20 x 4
N ellipses per cell = 5
Aspect ratios = [1.0, 1.25, 1.5, 1.75, 2.0]
Ellipse heights (µm) = [50.0, 62.5, 75.0, 87.5, 100.0]
Unit cell height = 475.0 µm
pitchX = 75 µm, pitchY = 500.0 µm
Starting layout export...
...gds generation
...Wrote file './ellipse_array_20x4_w50um_AR1-2_N5_20260317.gds'


GDS exported: ellipse_array_20x4_w50um_AR1-2_N5_20260317.gds
